# Battery Capacity Predictor: The Robot Guessing Game

This notebook demonstrates how we can train a **Robot Friend** (an Extra Trees Regressor) to predict battery capacity based on manufacturing parameters from the calendering step.

### Step 1: Packing the Robot's Toolbox (Imports)

**Robot Story:** Before the robot can start, it needs its tools. 
* **Pandas & Numpy:** For reading the big book of data and doing math.
* **Matplotlib:** For drawing pictures of its progress.
* **Sklearn:** This is the 'Brain Store' where we buy the robot's thinking parts.
* **SHAP:** This is the 'Truth Serum' magnifying glass to see inside the robot's head.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import RepeatedKFold, cross_val_score, cross_val_predict
from sklearn.preprocessing import MinMaxScaler
from numpy import mean, std

# Constants for easy configuration
DATA_FILE = "Calendering Step of Manufacturing.csv"
FEATURE_START = 'RollGap'
FEATURE_END = 'Coating_weight'
TARGET_COLUMN_INDEX = 10
NUM_ESTIMATORS = 1000
NUM_CV_SPLITS = 5
NUM_RUNS = 2

### Step 2: Cleaning the Clues (Data Loading and Preprocessing)

**Robot Story:** The robot needs to read the history book and make all the clues (features) fair using a 'Shrink Ray' (MinMaxScaler).

In [ ]:
def load_and_preprocess_data(file_path):
    """Loads CSV and scales features using MinMaxScaler."""
    # Robot Story: Read the book!
    data = pd.read_csv(file_path)
    
    # Robot Story: Pick the clues!
    feature_data = data.loc[:, FEATURE_START:FEATURE_END]
    feature_names = feature_data.columns
    
    # Robot Story: Use the Shrink Ray so every clue is equally big (between 0 and 1)!
    scaler = MinMaxScaler()
    scaled_values = scaler.fit_transform(feature_data.values)
    scaled_features_df = pd.DataFrame(scaled_values, columns=feature_names)
    
    return data, scaled_features_df

# Execute loading
try:
    raw_data, scaled_features = load_and_preprocess_data(DATA_FILE)
    print("Data loaded and clues cleaned!")
except FileNotFoundError:
    print(f"Warning: Could not find '{DATA_FILE}'. Please make sure it's in the same folder.")

### Step 3: Grading the Robot (Model Evaluation Logic)

**Robot Story:** We have three teachers (RMSE, MAE, R2) who grade the robot's performance.

In [ ]:
def evaluate_model(model, features, target, cv_strategy):
    """Calculates RMSE, MAE, and R2 scores using cross-validation."""
    # RMSE Teacher: Punishes big mistakes!
    rmse_scores = cross_val_score(
        estimator=model, X=features, y=target, 
        scoring='neg_root_mean_squared_error', cv=cv_strategy, n_jobs=-1
    )
    
    # MAE Teacher: Checks the average mistake size!
    mae_scores = cross_val_score(
        estimator=model, X=features, y=target, 
        scoring='neg_mean_absolute_error', cv=cv_strategy, n_jobs=-1
    )
    
    # R2 Teacher: Checks if the robot is actually smart!
    r2_scores = cross_val_score(
        estimator=model, X=features, y=target, 
        scoring='r2', cv=cv_strategy, n_jobs=-1
    )
    return rmse_scores, mae_scores, r2_scores

### Step 4: The Final Performance (Main Training and Visualization)

**Robot Story:** We play the game many times, make 'Official Homework Guesses', and look inside the robot's brain with the 'SHAP Magnifying Glass'.

In [ ]:
def main():
    # 1. Setup target answers
    target_name = raw_data.columns[TARGET_COLUMN_INDEX]
    target_values = raw_data[target_name]
    
    total_shap_values = 0
    
    # 2. Execution Loop: Play the whole game twice (NUM_RUNS)!
    for run_count in range(NUM_RUNS):
        # Use 1,000 tiny mini-brains for the robot!
        model = ExtraTreesRegressor(n_estimators=NUM_ESTIMATORS, bootstrap=False)
        
        # Hide-and-seek rules (5 piles, hide one, practice on the rest)!
        cv_strategy = RepeatedKFold(n_splits=NUM_CV_SPLITS, n_repeats=1)
        
        # The Teachers Grade the Robot
        rmse, mae, r2 = evaluate_model(model, scaled_features, target_values, cv_strategy)
        print(f"\nRun {run_count} Results:")
        print(f"Average Error (RMSE): {-mean(rmse):.3f}")
        print(f"Smartness Score (R2): {mean(r2):.3f}")
        
        # Making Official Homework Guesses
        predictions = cross_val_predict(model, scaled_features, target_values, cv=cv_strategy)
        
        # Drawing the 'Official Guess' Picture (Purple Dots vs Gold Star Line)
        fig, ax = plt.subplots(figsize=(4, 4))
        ax.scatter(target_values, predictions, color='purple', alpha=0.6)
        ax.plot([target_values.min(), target_values.max()], [target_values.min(), target_values.max()], 'k--', lw=2)
        ax.set_xlabel(f'Actual {target_name}')
        ax.set_ylabel(f'Predicted {target_name}')
        ax.set_title(f"Run {run_count}: Guesses vs Reality")
        plt.tight_layout()
        plt.show()
        
        # Deep Brain Inspection (SHAP Truth Serum)
        model.fit(scaled_features, target_values)
        explainer = shap.Explainer(model)
        total_shap_values += explainer.shap_values(scaled_features)

    # 3. Final Clue Summary (Which clue was the hero?)
    print("\nFinal Clue Importance Summary:")
    shap.summary_plot(total_shap_values / NUM_RUNS, scaled_features, plot_type="bar")

if __name__ == "__main__":
    if 'raw_data' in locals():
        main()
    else:
        print("Cannot run main() without raw_data. Please fix the data path.")